# 04 · Backtest: would the strategy have worked?
Proposal section 4. The Zurich banker starts at **35** and follows each strategy month by month.
At every month-end only past data is used (rolling 60 months → Ledoit-Wolf + Black-Litterman → γ re-calibrated → re-optimise).
Costs: 10 bp × turnover. Negative CHF rates included.

* **A** standard MVO · **B** 100 − age · **D** total wealth without the hedge · **C** our total-wealth portfolio
* (A) vs (D): "your job is bond-like, so take more risk" · (D) vs (C): "avoid your own sector"

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import backtest as bt
from config import INVESTABLE
from data_loader import load_excess_returns, load_rates
from personas import BANKER

p = BANKER
X, R = load_excess_returns(), load_rates()
t0 = time.time()
pre = bt.precompute(X, INVESTABLE)          # inputs + gamma for every month (no look-ahead)
res = bt.run_all(p, pre, X, R)
print(f"{pre.dates[0]} to {pre.dates[-1]}: {len(pre.dates)} months, {time.time() - t0:.0f}s")

def shade(ax):
    for a, b in bt.CRISES.values():
        ax.axvspan(pd.Period(a, "M").start_time, pd.Period(b, "M").end_time, color="grey", alpha=0.15)
COLORS = {"A": "tab:grey", "B": "tab:olive", "C": "tab:blue", "D": "tab:orange"}

## 1. The client's balance sheet during the backtest (strategy C)

In [ ]:
df = res["C"]
ax = (df[["H", "P", "F"]] / 1e6).to_timestamp().plot.area(figsize=(11, 4), alpha=0.85)
shade(ax); ax.set_ylabel("CHF million"); ax.set_title("Career H, pension P, portfolio F")

**Reading:** the career moves with interest rates (β_B ≈ 3): it jumps when rates fall (2008, 2019) and drops in 2022.
The portfolio is small next to it, so it can only change a small part of total risk.

## 2. Portfolio value under each strategy

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for s, d in res.items():
    (d["F"] / 1e3).to_timestamp().plot(ax=ax, label=bt.LABELS[s], color=COLORS[s])
shade(ax); ax.set_ylabel("CHF thousand"); ax.legend(); ax.set_title("Portfolio F (savings added every month)")

## 3. What each strategy holds over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, s in zip(axes, ["D", "C"]):
    w = res[s].filter(like="w_").rename(columns=lambda c: c[2:]).to_timestamp()
    w.plot.area(ax=ax, legend=(s == "C"), colormap="tab10")
    ax.set_title(bt.LABELS[s]); ax.set_ylim(0, 1)
axes[1].legend(bbox_to_anchor=(1, 1)); plt.tight_layout()

**Reading:** D buys the bank sector whenever it looks cheap; C never does, because the banker already owns it through his job.

## 4. The scorecard

In [ ]:
M = bt.metrics(res)
fmt = {
    "CE return on total wealth": "{:.2%}", "CE gain vs A (CHF/yr)": "{:+,.0f}",
    "corr(portfolio, sector part of career)": "{:.2f}",
    "avg portfolio P&L in worst 10% sector months (CHF)": "{:+,.0f}",
    "vol of total wealth": "{:.1%}", "max drawdown of total wealth": "{:.1%}",
    "vol of portfolio": "{:.1%}", "max drawdown of portfolio": "{:.1%}",
    "turnover per year": "{:.0%}", "tracking error vs A": "{:.1%}", "final portfolio F (CHF)": "{:,.0f}",
}
M.apply(lambda row: row.map(fmt[row.name].format), axis=1)

**How to read the scorecard** (headline = certainty-equivalent return on total wealth, what the strategy maximises):
* **A → D** is the big step: taking more equity risk because the job is bond-like. Its price is visible in the larger portfolio drawdowns.
* **D → C** removes the bank sector. Check two rows: the **correlation with the sector part of the career** (lower for C)
  and the **CE return** (in this one historical path D can come out ahead, because the banks it bought after crashes rebounded;
  section 6 separates the risk effect from this return effect).
* Total-wealth volatility moves little because the career dominates wealth (proposal 4.1).

## 5. Crises, in CHF (computed in the backtest, never assumed)

In [ ]:
pd.concat({"portfolio P&L": bt.crisis_table(res, "pnl_F")}, axis=1).style.format("{:+,.0f}")

**Reading:** C and D lose more than A in crises because they hold more equity (the price of the first effect, shown, not hidden).
Compare **C with D** in the banking crises (2008, 2011): C does not hold the banker's own sector.
In equity-wide crises (COVID, 2022) C can lose more, because it holds more world equities.

## 6. Robustness: what if our β_S is wrong? (proposal 4.2)

In [ ]:
wb = bt.wrong_beta_curve(p, pre, X, R)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
ax = axes[0]
mult = wb.index.values
ax.plot(mult, wb["risk benefit C vs D (bp/yr)"], "o-", lw=2, label="risk benefit (lower variance)")
ax.plot(mult, wb["return effect C vs D (bp/yr)"], "s--", label="return effect (this historical path)")
ax.plot(mult, wb["CE gain C vs D (bp/yr)"], "^:", color="black", label="total CE gain")
ax.axhline(0, color="grey", lw=0.8); ax.legend()
ax.set_xlabel("true β_S / assumed β_S"); ax.set_ylabel("bp per year"); ax.set_title("Wrong-beta curve: C vs D")

ax = axes[1]
grid = np.linspace(0, 2, 41)
h = p.beta_S
ax.plot(grid, bt.theoretical_hedge_gain(h, grid * h) / h**2, color="black", label="theory h(2β − h)")
ax.plot(mult, [bt.sample_hedge_gain(X[p.sector_asset], h, m_ * h) / h**2 for m_ in mult], "o", label="measured on our data")
ax.axhline(0, color="grey", lw=0.8); ax.legend()
ax.set_xlabel("true β_S / assumed β_S"); ax.set_ylabel("variance reduction / h²"); ax.set_title("Code check of the formula")
plt.tight_layout()
wb.round(1)

**Reading:**
* **Left, solid line:** the risk benefit of C over D **grows with the true sector beta**, as the theory says: the more the career
  really moves with banks, the more it pays not to hold banks. It stays positive even at β = 0 because long-only caps the hedge
  (C can hold 0% banks, never less), so C is never over-hedged: the corner solutions make the strategy *more* robust (proposal 4.2).
* **Left, dashed line:** the return effect is negative in this sample: D bought European banks after crashes and they rebounded.
  That is one historical path, not something the strategy is designed to predict.
* **Right:** the formula h(2β − h) checked on the data: zero gain when the true beta is half the assumed one, a pure cost when it is zero,
  75% of the maximum when it is double. Robust to underestimating β_S, fragile to overestimating it, hence the conservative presets.

## 7. Robustness: job loss in banking crises

In [ ]:
rows = {}
base = {s: bt.certainty_equivalent(res[s]["r_W"], np.mean(res["A"]["gamma"])) for s in "CD"}
rows["no job loss"] = {"job losses start": "-", "C minus D (bp/yr)": 1e4 * (base["C"] - base["D"])}
for th in [0.20, 0.30, 0.40]:
    r = {s: bt.run(s, p, pre, X, R, job_loss_threshold=th) for s in "ABCD"}
    g = np.mean(r["A"]["gamma"])
    ce = {s: bt.certainty_equivalent(r[s]["r_W"], g) for s in r}
    u = r["C"]["employed"]
    starts = [str(d) for d, a, b in zip(u.index, u.shift(fill_value=True), u) if a and not b]
    rows[f"{th:.0%} sector drop"] = {"job losses start": ", ".join(starts),
                                     "C minus D (bp/yr)": 1e4 * (ce["C"] - ce["D"]),
                                     **{bt.LABELS[s]: ce[s] for s in "ABCD"}}
jl = pd.DataFrame(rows).T
jl

**Reading:** the banker loses 12 months of salary when his sector falls more than the threshold below its 2-year high.
Job losses land in the banking crises. Look at the "C minus D" column: when the career shock is real (job loss),
the difference moves in C's favour compared with the no-job-loss case.

## Honesty note (to say in the app and the pitch)
A backtest is **one** historical path, and part of the career return is **modelled, not observed**.
This backtest tests the logic and robustness of the strategy, not a guaranteed alpha.